In [3]:
!pip install groq python-dotenv

In [5]:
import os
from dotenv import load_dotenv
from groq import Groq

load_dotenv() 

client = Groq(api_key=os.getenv("GROQ_API_KEY"))

response = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[{"role": "user", "content": "Say hello in one short sentence."}]
)

print(response.choices[0].message.content)

Hello!


In [6]:
from Clinical_Ontology import CHIEF_COMPLAINT_ONTOLOGY, STANDARD_HISTORY_SECTIONS

In [7]:
def detect_chief_complaint(patient_text):
    
    known_complaints = list(CHIEF_COMPLAINT_ONTOLOGY.keys())
    
    prompt = f"""The patient said: "{patient_text}"
    Which of these known complaints does this best match: "{known_complaints}"?
    Respond only with the matching category name from the list, or "unknown" if none fit. No Explaination, just the category name"""

    response = client.chat.completions.create(
        model = "openai/gpt-oss-120b",
        messages = [{"role": "user", "content": prompt}],
        temperature = 0
    )
    return response.choices[0].message.content.strip()

In [8]:
test_result = detect_chief_complaint("I've had really bad chest pain since morning")
print(test_result)

chest_pain


In [17]:
conversation_state = {
    "cheif_complaint": None,
    "current_section": "chief_complaint",
    "question_index": 0,
    "transcript": []
}

In [18]:
def get_next_question(state):
    
    if state["current_section"] == "chief_complaint":
        questions = CHIEF_COMPLAINT_ONTOLOGY[state["chief_complaint"]]["follow_up_questions"]
        
        if state["question_index"] < len(questions):
            return questions[state["question_index"]]
        else:
            state["current_section"] = "past_medical_surgical_history"
            state["question_index"] = 0
            return get_next_question(state)
    
    elif state["current_section"] in STANDARD_HISTORY_SECTIONS:
        questions = STANDARD_HISTORY_SECTIONS[state["current_section"]]
        
        if state["question_index"] < len(questions):
            return questions[state["question_index"]]
        else:
            section_order = list(STANDARD_HISTORY_SECTIONS.keys())
            current_idx = section_order.index(state["current_section"])
            
            if current_idx + 1 < len(section_order):
                state["current_section"] = section_order[current_idx + 1]
                state["question_index"] = 0
                return get_next_question(state)
            else:
                state["current_section"] = "done"
                return None
    
    return None

In [19]:
conversation_state["chief_complaint"] = "chest_pain"

q1 = get_next_question(conversation_state)
print(q1)

Where exactly is the pain located?
